# Notebook 05 — Risk Assessment Agent Training
This notebook trains the Risk Assessment Agent's model using Logistic Regression fused with Platt scaling (via `CalibratedClassifierCV`).

### Research & Design Rationale:
1. **Interpretable Fusion:** Logistic regression coefficients are directly reportable to regulators, unlike black-box deep learning models.
2. **Speed & Efficiency:** Microsecond inference vs. 100ms+ for neural alternatives.
3. **Platt Calibration:** Calibrating probability output using Platt scaling ensures `risk_score = 0.8` means 80% of such inputs are actually malicious, which is a prerequisite for reliable governance rules.
4. **Data Leakage Prevention:** The `StandardScaler` is fitted *only* on the training set.

In [ ]:
import os, sys, json, pickle, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, RocCurveDisplay, brier_score_loss
)
from sklearn.model_selection import GridSearchCV

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

COLAB_BASE = "/content/drive/MyDrive/PAS"
if os.path.exists(COLAB_BASE):
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    BASE = COLAB_BASE
else:
    if os.path.isdir("NOTEBOOKS"):
        BASE = str(Path(".").resolve())
    elif os.path.isdir("../NOTEBOOKS"):
        BASE = str(Path("..").resolve())
    else:
        BASE = str(Path(".").resolve().parent)
import sys
sys.path.insert(0, os.path.join(BASE, "AGENTS"))
COLAB_BASE = "/content/drive/MyDrive/PAS"
if os.path.exists(COLAB_BASE):
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    BASE = COLAB_BASE
else:
    if os.path.isdir("NOTEBOOKS"):
        BASE = str(Path(".").resolve())
    elif os.path.isdir("../NOTEBOOKS"):
        BASE = str(Path("..").resolve())
    else:
        BASE = str(Path(".").resolve().parent)
import sys
sys.path.insert(0, os.path.join(BASE, "AGENTS"))
COLAB_BASE = "/content/drive/MyDrive/PAS"
if os.path.exists(COLAB_BASE):
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    BASE = COLAB_BASE
else:
    if os.path.isdir("NOTEBOOKS"):
        BASE = str(Path(".").resolve())
    elif os.path.isdir("../NOTEBOOKS"):
        BASE = str(Path("..").resolve())
    else:
        BASE = str(Path(".").resolve().parent)
import sys
sys.path.insert(0, os.path.join(BASE, "AGENTS"))
COLAB_BASE = "/content/drive/MyDrive/PAS"
if os.path.exists(COLAB_BASE):
    from google.colab import drive; drive.mount("/content/drive", force_remount=False)
    BASE = COLAB_BASE
else:
    if os.path.isdir("NOTEBOOKS"):
        BASE = str(Path(".").resolve())
    elif os.path.isdir("../NOTEBOOKS"):
        BASE = str(Path("..").resolve())
    else:
        BASE = str(Path(".").resolve().parent)
import sys
sys.path.insert(0, os.path.join(BASE, "AGENTS"))
CACHE_DIR  = os.path.join(BASE, "DATASETS", "EXPERIMENT_CACHE")
if not os.path.exists(CACHE_DIR):
    CACHE_DIR = os.path.join(BASE, "EXPERIMENT_CACHE")

MODELS_DIR = os.path.join(BASE, "MODELS"); os.makedirs(MODELS_DIR, exist_ok=True)
RESULTS_DIR = os.path.join(BASE, "RESULTS"); os.makedirs(RESULTS_DIR, exist_ok=True)
METRICS_DIR = os.path.join(RESULTS_DIR, "metrics"); os.makedirs(METRICS_DIR, exist_ok=True)

print(f"BASE: {BASE}")
print(f"CACHE_DIR: {CACHE_DIR}")
print(f"MODELS_DIR: {MODELS_DIR}")
print(f"RESULTS_DIR: {RESULTS_DIR}")
print("Setup OK")

In [ ]:
features_path = os.path.join(CACHE_DIR, "features.parquet")
col_path = os.path.join(MODELS_DIR, "feature_columns.json")

assert os.path.exists(features_path), f"Run Notebook 04 first! Not found: {features_path}"
assert os.path.exists(col_path), f"feature_columns.json not found: {col_path}"

df = pd.read_parquet(features_path)
with open(col_path) as f:
    FEATURE_COLS = json.load(f)["columns"]

print(f"Loaded features shape: {df.shape}")
print(f"Feature columns: {FEATURE_COLS}")

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

X_train = train_df[FEATURE_COLS].values
y_train = train_df["label_int"].values
X_val   = val_df[FEATURE_COLS].values
y_val   = val_df["label_int"].values
X_test  = test_df[FEATURE_COLS].values
y_test  = test_df["label_int"].values

print(f"Train: {X_train.shape} (Malicious: {y_train.mean():.2%})")
print(f"Val  : {X_val.shape} (Malicious: {y_val.mean():.2%})")
print(f"Test : {X_test.shape} (Malicious: {y_test.mean():.2%})")

In [ ]:
# Scale features (fit ONLY on train)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print("Scaling done (train parameters applied to val/test to prevent leakage).")

# Grid search for best C on base estimator
param_grid = {"C": [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]}
lr_base = LogisticRegression(solver="lbfgs", max_iter=1000, random_state=RANDOM_SEED, class_weight="balanced")
grid = GridSearchCV(lr_base, param_grid, cv=3, scoring="f1_macro", n_jobs=-1, verbose=1)
grid.fit(X_train_scaled, y_train)

best_C = grid.best_params_["C"]
print(f"Best C found: {best_C} (CV F1-macro: {grid.best_score_:.4f})")

# Fit Platt scaling on top of the best base estimator
lr_best = LogisticRegression(C=best_C, solver="lbfgs", max_iter=1000, random_state=RANDOM_SEED, class_weight="balanced")
calibrated_lr = CalibratedClassifierCV(lr_best, cv=3, method="sigmoid")
calibrated_lr.fit(X_train_scaled, y_train)

print("Calibrated model successfully fit on training data.")

In [ ]:
# Detailed evaluation
for split_name, X_s, y_s in [("val", X_val_scaled, y_val), ("test", X_test_scaled, y_test)]:
    y_pred  = calibrated_lr.predict(X_s)
    y_proba = calibrated_lr.predict_proba(X_s)[:, 1]
    
    # Calculate key metrics
    accuracy = accuracy_score(y_s, y_pred)
    f1 = f1_score(y_s, y_pred, average="macro")
    auc = roc_auc_score(y_s, y_proba)
    fnr = ((y_s == 1) & (y_pred == 0)).sum() / max(1, (y_s == 1).sum())
    fpr = ((y_s == 0) & (y_pred == 1)).sum() / max(1, (y_s == 0).sum())
    brier = brier_score_loss(y_s, y_proba)
    
    print(f"\n{'='*20} {split_name.upper()} SET EVALUATION {'='*20}")
    print(classification_report(y_s, y_pred, target_names=["benign", "malicious"]))
    print(f"F1-Macro:  {f1:.4f}")
    print(f"ROC-AUC:   {auc:.4f}")
    print(f"Brier Score: {brier:.4f}")
    print(f"False Negative Rate (FNR): {fnr:.2%}")
    print(f"False Positive Rate (FPR): {fpr:.2%}")

# 1. Feature Importance Plot
coefs = np.mean([clf.estimator.coef_[0] for clf in calibrated_lr.calibrated_classifiers_], axis=0)
importance = sorted(zip(FEATURE_COLS, np.abs(coefs)), key=lambda x: x[1], reverse=True)

fig, ax = plt.subplots(figsize=(10, 5))
names, vals = zip(*importance)
ax.barh(names[::-1], vals[::-1], color="#3498db", edgecolor="black")
ax.set_title("Risk Model Feature Importance (|Mean Coefficients|)", fontweight="bold")
ax.set_xlabel("|Coefficient Value|")
plt.tight_layout()
plt.savefig(os.path.join(METRICS_DIR, "05_feature_importance.png"), bbox_inches="tight", dpi=150)
plt.show()

# 2. Calibration Curve Plot
prob_true, prob_pred = calibration_curve(y_test, calibrated_lr.predict_proba(X_test_scaled)[:, 1], n_bins=10)
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(prob_pred, prob_true, marker='o', linewidth=2, label="Calibrated LR (Platt)")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfectly Calibrated")
ax.set_title("Calibration Curve (Reliability Diagram)", fontweight="bold")
ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Fraction of Positives")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(METRICS_DIR, "05_calibration_curve.png"), bbox_inches="tight", dpi=150)
plt.show()

# 3. ROC Curve Plot
fig, ax = plt.subplots(figsize=(6, 6))
RocCurveDisplay.from_predictions(y_test, calibrated_lr.predict_proba(X_test_scaled)[:, 1], ax=ax)
ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
ax.set_title("Receiver Operating Characteristic (ROC) Curve", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(METRICS_DIR, "05_roc_curve.png"), bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# Save model & scaler
with open(os.path.join(MODELS_DIR, "logistic_regression.pkl"), "wb") as f:
    pickle.dump(calibrated_lr, f)
with open(os.path.join(MODELS_DIR, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)
print(f"Model saved to: {os.path.join(MODELS_DIR, 'logistic_regression.pkl')}")
print(f"Scaler saved to: {os.path.join(MODELS_DIR, 'scaler.pkl')}")

# Compute risk scores for ALL samples to cache
all_features_scaled = scaler.transform(df[FEATURE_COLS].values)
all_probs = calibrated_lr.predict_proba(all_features_scaled)[:, 1]

risk_df = pd.DataFrame({
    "sample_id": df["sample_id"],
    "risk_score": all_probs,
    "split": df["split"]
})

# Categorize risk levels based on paper's threshold guidelines
def get_risk_level(score):
    if score < 0.35: return "LOW"
    elif score < 0.65: return "MEDIUM"
    elif score < 0.80: return "HIGH"
    else: return "CRITICAL"

risk_df["risk_level"] = risk_df["risk_score"].apply(get_risk_level)

risk_scores_path = os.path.join(CACHE_DIR, "risk_scores.csv")
risk_df.to_csv(risk_scores_path, index=False)
print(f"Risk scores saved to: {risk_scores_path} ({len(risk_df):,} samples)")

In [ ]:
# Verify saved model
with open(os.path.join(MODELS_DIR, "logistic_regression.pkl"), "rb") as f:
    reloaded_lr = pickle.load(f)
with open(os.path.join(MODELS_DIR, "scaler.pkl"), "rb") as f:
    reloaded_scaler = pickle.load(f)

test_feats_scaled = reloaded_scaler.transform(df[df["split"] == "test"][FEATURE_COLS].values)
reloaded_probs = reloaded_lr.predict_proba(test_feats_scaled)[:, 1]

# Assertion check to ensure matching predictions
np.testing.assert_array_almost_equal(
    reloaded_probs,
    risk_df[risk_df["split"] == "test"]["risk_score"].values,
    decimal=5
)
print("Validation Check passed! Reloaded model outputs match exactly.")